# SHAWAF Stage 1 — TVPReid text-to-video eval

Frozen VLM encoders (X-CLIP, LanguageBind, InternVideo2 CLIP-S) on the unofficial Hub mirror **[bassatbassat/TVPReid](https://huggingface.co/datasets/bassatbassat/TVPReid)**.

This notebook **imports** `shawaf_vlm`; it does not reimplement the eval loop.

**Kaggle:** enable **GPU** + **Internet**. After any CUDA crash, **Restart session** then **Run All from the top.** The install cell always `git fetch` + `reset --hard origin/main` into `/kaggle/working/shawaf-vlm` and must print `shawaf_vlm 0.1.6`. If it prints an older version, re-run that cell.

The notebook clones [this GitHub repo](https://github.com/BASSAT-BASSAT/Benchmarking-Video-Image-language-models-for-Tracklet-retrieval-) and downloads only the **test** mp4s from **[bassatbassat/TVPReid](https://huggingface.co/datasets/bassatbassat/TVPReid)** (not train/val). Cite Zhang et al. (ACM MM 2024) plus the PRID / iLIDS / Duke source papers. Drop `"duke"` from `SUBSETS` for a much smaller download.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/BASSAT-BASSAT/Benchmarking-Video-Image-language-models-for-Tracklet-retrieval-.git"
REPO_DIR = "/kaggle/working/shawaf-vlm"
PACKAGE_DIR = ""  # optional local / Kaggle dataset path that already contains shawaf_vlm

HF_REPO = "bassatbassat/TVPReid"
SPLIT = "test"
# Test-only download. Drop "duke" (~600 videos) for a faster smoke run.
SUBSETS = ["prid", "ilids", "duke"]

MODELS = ["languagebind", "internvideo2"]
RUN_CLIP_1B = False
if RUN_CLIP_1B:
    MODELS.append("internvideo2_clip_1b")

NUM_FRAMES = 8
BATCH_SIZE = 4
TEXT_BATCH_SIZE = 32
DEVICE = "cuda"

_kaggle_work = Path("/kaggle/working")
_work = _kaggle_work if _kaggle_work.is_dir() else Path.cwd()
FRAME_CACHE = _work / "tvpreid_frames"
RESULTS_DIR = _work / "results"

In [ ]:
import os
import shutil
import sys
import subprocess
from pathlib import Path


def _has_package(root: Path) -> bool:
    return (root / "shawaf_vlm").is_dir() and (root / "pyproject.toml").is_file()


def find_repo_root() -> Path:
    kaggle = Path("/kaggle/working").is_dir()
    repo = Path(REPO_DIR)

    # On Kaggle always refresh /kaggle/working/shawaf-vlm. cwd already contains
    # shawaf_vlm after the first run, so a local-first search would skip git pull.
    if kaggle:
        if (repo / ".git").is_dir():
            subprocess.check_call(
                ["git", "-C", str(repo), "fetch", "--depth", "1", "origin", "main"]
            )
            subprocess.check_call(
                ["git", "-C", str(repo), "reset", "--hard", "origin/main"]
            )
            return repo
        if repo.exists():
            shutil.rmtree(repo)
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(repo)])
        return repo

    if PACKAGE_DIR and _has_package(Path(PACKAGE_DIR)):
        return Path(PACKAGE_DIR)
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if _has_package(candidate):
            return candidate
    if _has_package(repo):
        return repo
    raise RuntimeError(f"shawaf_vlm not found. Clone {REPO_URL}")


repo = find_repo_root()
os.chdir(repo)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", ".[all]"])

# Drop stale imports from an earlier notebook run.
for name in list(sys.modules):
    if name == "shawaf_vlm" or name.startswith("shawaf_vlm."):
        del sys.modules[name]
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

import shawaf_vlm

print("package root", repo, flush=True)
print("shawaf_vlm", shawaf_vlm.__version__, shawaf_vlm.__file__, flush=True)

In [ ]:
from shawaf_vlm.data.tvpreid import (
    discover_local_tvpreid,
    download_tvpreid,
    load_tvpreid_from_root,
)

local_root = discover_local_tvpreid()
if local_root is None:
    print("Downloading", HF_REPO, "split", SPLIT, "subsets", SUBSETS, flush=True)
    tvpreid_root = download_tvpreid(
        repo_id=HF_REPO,
        configs=tuple(SUBSETS),
        split=SPLIT,
        max_workers=8,
    )
else:
    tvpreid_root = local_root
    print("Using local TVPReid at", tvpreid_root, flush=True)

FRAME_CACHE.mkdir(parents=True, exist_ok=True)
splits_by_subset = {}
for subset in SUBSETS:
    splits = load_tvpreid_from_root(tvpreid_root, config=subset, split=SPLIT)
    splits_by_subset[subset] = splits
    print(
        f"{subset:6}  queries={len(splits.query):4}  "
        f"gallery={len(splits.gallery):4}  source={splits.source}"
    )

In [ ]:
import json
from pathlib import Path

import torch

from shawaf_vlm.eval_loop import evaluate_text_to_tracklet
from shawaf_vlm.metrics import format_metrics
from shawaf_vlm.models import all_specs, build_encoder
from shawaf_vlm.models.runtime import ensure_cuda_healthy

print("torch", torch.__version__, "cuda", torch.cuda.is_available(), flush=True)
if DEVICE.startswith("cuda") and not torch.cuda.is_available():
    raise RuntimeError("Kaggle GPU is off. Turn on GPU and rerun from the top.")
if torch.cuda.is_available():
    print("GPU", torch.cuda.get_device_name(0), flush=True)
ensure_cuda_healthy(DEVICE)

results = []
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

for name in MODELS:
    spec = all_specs()[name]
    print("\n===", spec.label, "===")
    encoder = build_encoder(name, device=DEVICE)
    for subset, splits in splits_by_subset.items():
        print(f"\n--- {subset} ---")
        metrics = evaluate_text_to_tracklet(
            encoder=encoder,
            splits=splits,
            num_frames=NUM_FRAMES,
            batch_size=BATCH_SIZE,
            text_batch_size=TEXT_BATCH_SIZE,
            frame_cache=FRAME_CACHE,
        )
        print(format_metrics(metrics))
        payload = {
            "model_key": spec.key,
            "model": spec.label,
            "checkpoint": spec.checkpoint,
            "dataset": "tvpreid",
            "hub_repo": HF_REPO,
            "subset": subset,
            "split": SPLIT,
            "split_source": splits.source,
            "num_queries": len(splits.query),
            "num_gallery": len(splits.gallery),
            "num_frames": NUM_FRAMES,
            "metrics": metrics,
        }
        path = RESULTS_DIR / f"{name}_tvpreid_{subset}.json"
        path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
        results.append(payload)
    del encoder

In [ ]:
print(f"{'model':<22} {'subset':<8} {'R1':>7} {'R5':>7} {'R10':>7} {'mAP':>7} {'Q':>5}")
for row in results:
    m = row["metrics"]
    print(
        f"{row['model_key']:<22} {row['subset']:<8} "
        f"{m['Rank-1']:7.2f} {m['Rank-5']:7.2f} {m['Rank-10']:7.2f} "
        f"{m['mAP']:7.2f} {int(m['num_valid_queries']):5d}"
    )